In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ete3 import Tree
from sklearn.linear_model import LinearRegression

plt.rcParams['pdf.fonttype'] = 42

plt.rcParams['ps.fonttype'] = 42

plt.rcParams['font.family'] = 'sans-serif'

plt.rcParams['font.sans-serif'] = 'Arial'


# -------------------------------------------------------------
# 1. FASTA / GENOTYPE PROCESSING
# -------------------------------------------------------------
def load_fasta_as_dataframe(fasta_file):
    df = pd.read_csv(
        fasta_file,
        lineterminator=">",
        engine="c",
        skiprows=1,
        names=["raw"],
    )
    df[["col0", "col1"]] = pd.DataFrame(
        df.raw.apply(lambda s: s.split(maxsplit=1)).tolist()
    )
    df["col1"] = df["col1"].str.replace("\n", "")
    return df


def genotypes_to_dict(mut_fa_df):
    genome_dict = {}
    for i in range(len(mut_fa_df)):
        barcode = mut_fa_df.iloc[i]["col0"][8:-2]
        genotype = mut_fa_df.iloc[i]["col1"]
        genome_dict[barcode] = (i, genotype)
    return genome_dict


# -------------------------------------------------------------
# 2. GENOTYPE SIMILARITY
# -------------------------------------------------------------
def similarity_between_strings(a, b):
    matches = 0
    effective_len = len(a)

    for i, ch in enumerate(a):
        if ch == "-":
            effective_len -= 1
            continue
        if ch == b[i]:
            matches += 1

    return matches / effective_len * 100 if effective_len > 0 else 0


def similarity_matrix(genome_dict):
    keys = list(genome_dict.keys())
    n = len(keys)

    sim = np.zeros((n, n))

    for i, k1 in enumerate(keys):
        seq1 = genome_dict[k1][1]
        for j, k2 in enumerate(keys):
            seq2 = genome_dict[k2][1]
            sim[i, j] = similarity_between_strings(seq1, seq2)

    return sim, keys


# -------------------------------------------------------------
# 3. TREE DISTANCES
# -------------------------------------------------------------
def tree_distance_matrix(tree, genome_dict):
    sorted_items = sorted(genome_dict.items(), key=lambda x: x[1][0])
    barcodes = [x[0] for x in sorted_items]

    leaf_map = {str(leaf)[11:-2]: leaf for leaf in tree.iter_leaves()}

    n = len(barcodes)
    dist = np.zeros((n, n))

    for i, b1 in enumerate(barcodes):
        leaf1 = leaf_map[b1]
        for j, b2 in enumerate(barcodes):
            leaf2 = leaf_map[b2]
            dist[i, j] = leaf1.get_distance(leaf2)

    return dist, barcodes


# -------------------------------------------------------------
# 4. PAIRWISE LISTS
# -------------------------------------------------------------
def matrix_upper_triangle_to_lists(sim_matrix, dist_matrix):
    sim_list, dist_list = [], []
    n = sim_matrix.shape[0]

    for i in range(n):
        for j in range(i, n):
            sim_list.append(sim_matrix[i, j])
            dist_list.append(dist_matrix[i, j])

    return sim_list, dist_list


# -------------------------------------------------------------
# 5. PLOTTING
# -------------------------------------------------------------
def plot_similarity_vs_distance(sim_list, dist_list, output_name):
    plt.figure(figsize=(6, 6))

    plt.plot(
        sim_list,
        dist_list,
        "*",
        markersize=0.2,
        color="#4F71BE",
        alpha=0.15,
        rasterized = False
    )

    plt.xlabel("Similarity between genotype pairs (%)", fontsize=16)
    plt.ylabel("Distance in the tree (subs/site)", fontsize=16)
    plt.title("Genotype similarity over the tree", fontsize=16)

    plt.xlim(0, 119)

    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    output_pdf = f'{output_name}.pdf'
    output_png = f'{output_name}.png'
    output_eps = f'{output_name}.eps'
    plt.savefig(output_pdf, dpi=600, bbox_inches='tight')
    plt.savefig(output_eps, dpi=600, bbox_inches='tight')
    plt.savefig(output_png, dpi=1000, bbox_inches='tight')
    plt.close()


# -------------------------------------------------------------
# MAIN FUNCTION 
# -------------------------------------------------------------
def compute_similarity_over_tree(tree_file, fasta_file, output_name="genotype_similarity_over_tree"):

    tree = Tree(open(tree_file).read(), format=1)
    mut_fa = load_fasta_as_dataframe(fasta_file)
    genome_dict = genotypes_to_dict(mut_fa)

    sim_matrix, _ = similarity_matrix(genome_dict)
    dist_matrix, _ = tree_distance_matrix(tree, genome_dict)

    sim_list, dist_list = matrix_upper_triangle_to_lists(sim_matrix, dist_matrix)

    plot_similarity_vs_distance(sim_list, dist_list, output_name)

    print(f"Saved PDF: {output_name}")


In [3]:
compute_similarity_over_tree(
    "fasttree_no_pon",
    "mutations.fa",
    output_name="genotypes_similarity_over_tree_no_rest"
)


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Saved PDF: genotypes_similarity_over_tree_no_rest
